# UniLife - Erasmus Mobility and Economic Gaps

This notebook documents the core UniLife pipeline: Erasmus+ KA1 Higher Education study flows, economic indicators, and descriptive analysis. The project is exploratory and does not claim causal inference.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path('..').resolve()
sys.path.append(str(PROJECT_ROOT / 'src'))

from unilife.ka1 import build_ka1_he_studies_flows
from unilife.indicators import fetch_country_indicators, indicator_coverage, join_indicators

DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

## 1. Build KA1 Higher Education Study Flows

In [ ]:
flows, flow_diagnostics = build_ka1_he_studies_flows(PROJECT_ROOT / 'data/raw/Erasmus-KA1-Mobility-Data-2022.xlsx')
flows.head(), flow_diagnostics

In [ ]:
flows.shape, flows['flow_count'].sum(), (flows['sending_code'] == flows['receiving_code']).sum()

## 2. Italy and Luxembourg Check

In [ ]:
flows.loc[flows['sending_code'].isin(['IT', 'LU'])]\
    .sort_values(['sending_code', 'flow_count'], ascending=[True, False])\
    .groupby('sending_code')\
    .head(10)

## 3. Add Economic Indicators

In [ ]:
indicators = fetch_country_indicators(year=2022)
joined = join_indicators(flows, indicators)
coverage = indicator_coverage(joined)
coverage

In [ ]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
flows.to_csv(DATA_PROCESSED / 'ka1_he_studies_flows_long.csv', index=False)
flow_diagnostics.to_csv(DATA_PROCESSED / 'ka1_he_studies_filter_diagnostics.csv', index=False)
indicators.to_csv(DATA_PROCESSED / 'country_indicators_2022.csv', index=False)
joined.to_csv(DATA_PROCESSED / 'ka1_he_studies_flows_with_indicators.csv', index=False)
coverage.to_csv(DATA_PROCESSED / 'indicator_coverage_diagnostics.csv', index=False)

## 4. Descriptive Starting Points

In [ ]:
top_destinations = joined.groupby(['receiving_code', 'receiving_country'], as_index=False)['flow_count'].sum()
top_destinations.sort_values('flow_count', ascending=False).head(15)

In [ ]:
joined.dropna(subset=['gdp_gap_usd_wb'])[['sending_code', 'receiving_code', 'flow_count', 'gdp_gap_usd_wb', 'price_level_gap', 'earnings_gap']].head()